# Lagos Property Data — Data Cleaning
**Project step 2: Data cleaning with Pandas**

Input: `lagos_properties_raw.csv` (scraped from nigeriapropertycentre.com)
Output: `lagos_properties_clean.csv`

**All prices end up in Nigerian naira (₦).** Rent prices are treated as **annual** rent, the normal way Lagos rents are quoted.

## Step 1: Import libraries and set the exchange rate

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Official CBN rate on 8 Oct 2026 (Aboki Forex via Techeconomy), rounded
USD_TO_NGN = 1332

## Step 2: Load the raw data and take a first look
Before changing anything, check the size, the data types and how much is missing in each column.

In [2]:
raw = pd.read_csv("lagos_properties_raw.csv")
df = raw.copy()          # keep the raw data untouched for comparison

print("Shape:", df.shape)
print()
print(df.dtypes)

Shape: (2038, 18)

listing_id         int64
listing_type         str
title                str
property_type        str
price_raw            str
currency             str
price                str
price_unit       float64
location             str
bedrooms         float64
bathrooms        float64
toilets          float64
parking          float64
size_sqm         float64
added                str
url                  str
page               int64
scraped_on           str
dtype: object


In [3]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1)
})
missing[missing["missing"] > 0]

,missing,missing_%
price_unit,2038,100.0
bedrooms,281,13.8
bathrooms,250,12.3
toilets,229,11.2
parking,1081,53.0
size_sqm,1893,92.9
added,1128,55.3


In [4]:
df.head()

,listing_id,listing_type,title,property_type,price_raw,currency,price,price_unit,location,bedrooms,bathrooms,toilets,parking,size_sqm,added,url,page,scraped_on
0,3388814,Sale,Newly Built 4 Bedroom Fully Detached Bungalow,Detached bungalow,"₦100,000,000",₦,"100,000,000",NaN,"Igando Orudu, Ibeju Lekki, Lagos",4.0,4.0,5.0,3.0,NaN,NaN,https://nigeriapropertycentre.com/for-sale/hou...,1,2026-10-10
1,3087107,Sale,Land,Residential land,"₦50,000,000",₦,"50,000,000",NaN,"Mainland Heritage, 10mins Drive to Command Off...",NaN,NaN,NaN,5.0,500.0,NaN,https://nigeriapropertycentre.com/for-sale/lan...,1,2026-10-10
2,3715040,Sale,Residential Land Plot Fenced,Residential land,"₦75,000,000",₦,"75,000,000",NaN,"Off Ezobi Street, Alapere, Ketu, Lagos",NaN,NaN,NaN,NaN,481.0,NaN,https://nigeriapropertycentre.com/for-sale/lan...,1,2026-10-10
3,3663484,Sale,4 Bedroom Detached Duplex,Detached duplex,"₦280,000,000",₦,"280,000,000",NaN,"Idado, Lekki, Lagos",4.0,4.0,5.0,NaN,NaN,NaN,https://nigeriapropertycentre.com/for-sale/hou...,1,2026-10-10
4,3693891,Sale,Best Value 3 Bedroom Self Compound Terraced Du...,Terraced duplex,"₦115,000,000",₦,"115,000,000",NaN,"Within a Gated, Very Secured Estate with No Fl...",3.0,3.0,4.0,2.0,NaN,NaN,https://nigeriapropertycentre.com/for-sale/hou...,1,2026-10-10


## Step 3: Remove duplicates
Three checks, from strictest to loosest:
1. **Exact duplicate rows** (every column the same).
2. **Duplicate listing IDs** (the same listing scraped twice).
3. **Near-duplicates**: the same property re-posted under a new ID. Same title, price, location and sale/rent type means the same property.

In [5]:
before = len(df)

exact = df.duplicated().sum()
ids = df["listing_id"].duplicated().sum()
key = ["title", "price", "location", "listing_type"]
near = df.duplicated(subset=key).sum()
print(f"Exact duplicate rows:   {exact}")
print(f"Duplicate listing IDs:  {ids}")
print(f"Near-duplicates:        {near}")

df = df.drop_duplicates()
df = df.drop_duplicates(subset="listing_id")
df = df.drop_duplicates(subset=key, keep="first")
print(f"\nRows: {before} -> {len(df)}  (removed {before - len(df)})")

Exact duplicate rows:   0
Duplicate listing IDs:  0
Near-duplicates:        24

Rows: 2038 -> 2014  (removed 24)


In [6]:
# Example of a near-duplicate pair that was removed
raw[raw.duplicated(subset=key, keep=False)].sort_values("title")[key + ["listing_id"]].head(6)

,title,price,location,listing_type,listing_id
1857,1 Bedroom Apartment with Snooker,"15,000,000","Lekki Phase 1, Lekki, Lagos",Rent,3690826
1097,1 Bedroom Apartment with Snooker,"15,000,000","Lekki Phase 1, Lekki, Lagos",Rent,3710729
1288,2 Bedroom Apartment,"10,000,000","Ikate, Lekki, Lagos",Rent,3654614
1594,2 Bedroom Apartment,"10,000,000","Ikate, Lekki, Lagos",Rent,3718006
1582,2 Bedroom Apartment,"11,000,000","Ikate, Lekki, Lagos",Rent,3718050
1460,2 Bedroom Apartment,"8,500,000","Ikate, Lekki, Lagos",Rent,3718461


## Step 4: Convert price to a number and to naira
The scraped price is **text** (e.g. `"175,000,000"`), so it can't be averaged or sorted. Steps:
1. Remove the commas and convert to a number.
2. **Fix mislabelled currency.** A few listings show a dollar sign on a naira amount (e.g. a 5-bedroom house at **$4.5 billion**, which would be the most expensive home in Africa). Any "$" price of 100 million or more is treated as naira.
3. **Convert genuine US-dollar listings to naira** at ₦1,332/$.

In [7]:
df["price_original"] = df["price"].str.replace(",", "", regex=False).astype(float)
df["currency_original"] = df["currency"].map({"₦": "NGN", "$": "USD"})

mislabelled = (df["currency_original"] == "USD") & (df["price_original"] >= 100_000_000)
print("Dollar listings that are really naira:")
display(df.loc[mislabelled, ["title", "price_raw", "property_type"]])
df.loc[mislabelled, "currency_original"] = "NGN"

is_usd = df["currency_original"] == "USD"
df["price_ngn"] = np.where(is_usd, df["price_original"] * USD_TO_NGN, df["price_original"])

print(f"\nGenuine USD listings converted to naira: {is_usd.sum()}")
df.loc[is_usd, ["title", "price_raw", "price_ngn"]].head()

Dollar listings that are really naira:


,title,price_raw,property_type
118,Selling 5 Bedrooms Standalone Mansion,"$4,500,000,000",Detached duplex
399,4 Bedroom Apartment,"$800,000,000",Flat / apartment



Genuine USD listings converted to naira: 80


,title,price_raw,price_ngn
96,2 Bedroom Ultra-Luxury Apartment,"$700,000",9.324000e+08
123,"Residential Land Parcels, Different Sizes","$2,300",3.063600e+06
148,Exquisite 5-Bedroom Contemporary House,"$2,450,000",3.263400e+09
165,10th 4 Floor Luxury Penthouse,"$2,300,000",3.063600e+09
169,5 Bedroom Semi-Detached Duplex,"$4,000,000",5.328000e+09


## Step 5: Remove prices that are not a total price
The scraper did not capture the price **unit** (the `price_unit` column is empty), so some listings show a **per-square-metre** or **monthly** price instead of the full price. Examples: 1,000 sqm of land "for" ₦750,000, or a 650 sqm office "for" ₦100,000 a year.

These would wreck any average, so they are removed using clear rules:

| Rule | Reason |
|---|---|
| Sale price below ₦5M | No house or land plot in Lagos sells for that; these are per-sqm prices |
| Rent below ₦200k a year | Per-sqm or monthly prices |
| Original USD price below $10,000 | Per-sqm office rents and land prices quoted in dollars |
| Rent above ₦200M for a single home (≤ 8 bedrooms) | A sale price posted in the rent section (e.g. a 2-bed flat "renting" for ₦650M) |

In [8]:
rules = {
    "Sale < ₦5M":                  (df["listing_type"] == "Sale") & (df["price_ngn"] < 5_000_000),
    "Rent < ₦200k":                (df["listing_type"] == "Rent") & (df["price_ngn"] < 200_000),
    "USD price < $10,000":         (df["currency_original"] == "USD") & (df["price_original"] < 10_000),
    "Rent > ₦200M, single home":   (df["listing_type"] == "Rent") & (df["price_ngn"] > 200_000_000)
                                   & (df["bedrooms"].fillna(0) <= 8),
}
for name, mask in rules.items():
    print(f"{name:28} {mask.sum():>3} rows")

invalid = np.logical_or.reduce(list(rules.values()))
print("\nExamples of removed rows:")
display(df.loc[invalid, ["title", "price_raw", "listing_type", "property_type"]].head(8))

before = len(df)
df = df[~invalid]
print(f"Rows: {before} -> {len(df)}")

Sale < ₦5M                    18 rows
Rent < ₦200k                   7 rows
USD price < $10,000            6 rows
Rent > ₦200M, single home      9 rows

Examples of removed rows:


,title,price_raw,listing_type,property_type
20,A Well Planned Residential Estate with Good Am...,"₦4,000,000",Sale,Mixed-use land
24,Strategically Land Located in a Serene Environ...,"₦140,000",Sale,Mixed-use land
37,1144SQM Commercial Land,"₦1,000,000",Sale,Commercial land
63,Land Measuring 2000SQM,"₦1,800,000",Sale,Residential land
111,"6,615SQM Commercial Land at Oniru","₦1,300,000",Sale,Mixed-use land
123,"Residential Land Parcels, Different Sizes","$2,300",Sale,Residential land
159,"1, 480 SQM Prime Land","₦1,200,000",Sale,Residential land
251,"A Premium 1, 055SQM Land","₦450,000",Sale,Mixed-use land


Rows: 2014 -> 1975


## Step 6: Split location into neighbourhood and area, and standardise names
`location` is free text, e.g. `"Orchid Road, Lafiaji, Lekki, Lagos"`. The **area** (Lekki) is always the second-to-last part, and the **neighbourhood** (Lafiaji) the part before it.

- Area names are standardised (`"Victoria Island (VI)"` → `"Victoria Island"`).
- Rows whose location is not a real place (an agent's company name) are dropped.

In [9]:
parts = df["location"].str.split(",").apply(lambda p: [x.strip() for x in p])

df["area"] = parts.apply(lambda p: p[-2] if len(p) >= 2 else np.nan)
df["neighbourhood"] = parts.apply(lambda p: p[-3] if len(p) >= 3 else (p[-2] if len(p) >= 2 else np.nan))

bad_location = df["area"].isna()
print("Rows with no usable location:", bad_location.sum())
display(df.loc[bad_location, ["title", "location"]])
df = df[~bad_location]

df["area"] = df["area"].replace({"Victoria Island (VI)": "Victoria Island", "Isheri North": "Isheri"})
df["neighbourhood"] = df["neighbourhood"].str.title()

print("Areas:", df["area"].nunique())
df["area"].value_counts().head(10)

Rows with no usable location: 2


,title,location
393,Contemporary 5 Bedroom House,Prime Home Properties Lagos
1661,4 Bedroom Terraced Duplex,Prime Home Properties Lagos


Areas: 37


area
Lekki              919
Ikoyi              260
Ajah               204
Victoria Island    131
Ikeja              115
Ibeju Lekki         52
Gbagada             44
Magodo              41
Surulere            37
Yaba                15
Name: count, dtype: int64

## Step 7: Add an Island / Mainland column
Lagos property is usually discussed as **Island** (Lekki, Ikoyi, VI and the Lekki–Epe corridor) versus **Mainland** (everywhere else). This column is used for the headline comparison in the EDA.

In [10]:
ISLAND = ["Lekki", "Ajah", "Ikoyi", "Victoria Island", "Lagos Island",
          "Ibeju Lekki", "Eko Atlantic City", "Epe"]
df["region"] = np.where(df["area"].isin(ISLAND), "Island", "Mainland")
df["region"].value_counts()

region
Island      1569
Mainland     404
Name: count, dtype: int64

## Step 8: Standardise property types
The site uses 25 property types, with inconsistent capitalisation and some near-identical labels. Two cleaner columns are added:
- **`property_category`**: Residential, Land or Commercial (the broadest split).
- **`property_group`**: a simpler grouping for charts (Flat / Apartment, Duplex, Bungalow, House, Land, Commercial).

A few residential listings are tagged as "Commercial property" on the site, so the title is checked too.

In [11]:
df["property_type"] = df["property_type"].str.strip().str.capitalize()

def group(ptype, title):
    p, t = ptype.lower(), title.lower()
    if "land" in p:
        return "Land"
    if any(w in p for w in ["flat", "apartment", "self contain", "mini flat"]):
        return "Flat / Apartment"
    if "duplex" in p:
        return "Duplex"
    if "bungalow" in p:
        return "Bungalow"
    if p == "house":
        return "House"
    if p == "commercial property" and "duplex" in t:   # mislabelled homes
        return "Duplex"
    return "Commercial"

df["property_group"] = [group(p, t) for p, t in zip(df["property_type"], df["title"])]
df["property_category"] = df["property_group"].map(
    lambda g: g if g in ("Land", "Commercial") else "Residential")

pd.crosstab(df["property_group"], df["property_category"])

property_category,Commercial,Land,Residential
property_group,,,
Bungalow,0,0,19
Commercial,70,0,0
Duplex,0,0,804
Flat / Apartment,0,0,880
House,0,0,71
Land,0,129,0


## Step 9: Handle missing bedrooms
Missing bedrooms mean different things depending on the property:
- **Land and Commercial**: no bedrooms exist. These stay empty (`NaN`) on purpose, because filling them with 0 would wrongly drag down "average bedrooms".
- **Residential**: the number is often in the **title** ("Two Bedroom Apartment", "Studio"), so it is extracted from there.
- Residential rows still missing after that are filled with the **median for that property type**.

In [12]:
print("Missing bedrooms by category (before):")
print(df[df["bedrooms"].isna()]["property_category"].value_counts(), "\n")

WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
         "seven": 7, "eight": 8, "nine": 9, "ten": 10}

def beds_from_title(title):
    t = title.lower()
    if any(w in t for w in ["studio", "self contain", "single room", "bed space"]):
        return 1
    m = re.search(r"(\d+)\s*-?\s*bed", t)
    if m:
        return int(m.group(1))
    m = re.search(r"\b(" + "|".join(WORDS) + r")\s*-?\s*bed", t)
    if m:
        return WORDS[m.group(1)]
    return np.nan

res = df["property_category"] == "Residential"
need = res & df["bedrooms"].isna()
df.loc[need, "bedrooms"] = df.loc[need, "title"].apply(beds_from_title)
print("Filled from title:", need.sum() - (res & df["bedrooms"].isna()).sum())

still = res & df["bedrooms"].isna()
type_median = df[res].groupby("property_type")["bedrooms"].median()
df.loc[still, "bedrooms"] = df.loc[still, "property_type"].map(type_median)
print("Filled with property-type median:", still.sum() - (res & df["bedrooms"].isna()).sum())
print("Residential still missing:", (res & df["bedrooms"].isna()).sum())

Missing bedrooms by category (before):
property_category
Land           129
Residential     68
Commercial      54
Name: count, dtype: int64 

Filled from title: 47
Filled with property-type median: 21
Residential still missing: 0


## Step 10: Handle missing bathrooms and toilets
For residential homes, bathroom and toilet counts follow bedroom counts closely (a 4-bed duplex usually has 4 baths and 5 toilets). Missing values are filled with the **median for homes with the same number of bedrooms**. Land and commercial stay empty, as before.

In [13]:
for col in ["bathrooms", "toilets"]:
    before_missing = (res & df[col].isna()).sum()
    med_by_beds = df[res].groupby("bedrooms")[col].median()
    fill = df.loc[res & df[col].isna(), "bedrooms"].map(med_by_beds)
    df.loc[fill.index, col] = fill
    # any bedroom count with no reference homes: fall back to bedrooms
    left = res & df[col].isna()
    df.loc[left, col] = df.loc[left, "bedrooms"]
    print(f"{col:10} residential missing: {before_missing} -> {(res & df[col].isna()).sum()}")

bathrooms  residential missing: 47 -> 0
toilets    residential missing: 42 -> 0


## Step 11: Handle missing parking and drop unusable columns
- **Parking** is only shown when the agent fills it in. A missing value means "not stated", so it is set to **0** (no parking advertised).
- **Dropped columns:**

| Column | Why dropped |
|---|---|
| `price_unit` | 100% empty (scraper could not capture it) |
| `size_sqm` | 93% empty; only shown for land |
| `added` | 55% empty, and only says "today"/"yesterday", so no time trend is possible |
| `price`, `currency`, `price_raw` | Replaced by the clean `price_ngn` and `currency_original` |
| `page` | Scraping detail, not property data |

In [14]:
print("Parking missing:", df["parking"].isna().sum())
df["parking"] = df["parking"].fillna(0)

df = df.drop(columns=["price_unit", "size_sqm", "added", "price", "currency", "price_raw", "page"])
df.columns.tolist()

Parking missing: 1038


['listing_id',
 'listing_type',
 'title',
 'property_type',
 'location',
 'bedrooms',
 'bathrooms',
 'toilets',
 'parking',
 'url',
 'scraped_on',
 'price_original',
 'currency_original',
 'price_ngn',
 'area',
 'neighbourhood',
 'region',
 'property_group',
 'property_category']

## Step 12: Set correct data types and add helper columns
- Counts (bedrooms, bathrooms, toilets, parking) become **whole numbers**. `Int64` (capital I) allows empty values for land and commercial.
- `scraped_on` becomes a real **datetime**.
- Categories become the **category** type, which uses less memory and sorts properly.
- New columns that make analysis easier:
  - `price_ngn_m`: price in **millions of naira** (easier to read on charts).
  - `price_per_bedroom`: value-for-money measure for homes.
  - `is_multi_unit`: blocks, hotels and buildings with more than 8 rooms. These are investment buildings, not single homes.

In [15]:
for col in ["bedrooms", "bathrooms", "toilets", "parking"]:
    df[col] = df[col].round().astype("Int64")

df["listing_id"] = df["listing_id"].astype(str)
df["price_ngn"] = df["price_ngn"].round().astype("int64")
df["scraped_on"] = pd.to_datetime(df["scraped_on"])

df["price_ngn_m"] = (df["price_ngn"] / 1_000_000).round(2)
df["price_per_bedroom"] = np.where(df["bedrooms"].fillna(0) > 0,
                                   (df["price_ngn"] / df["bedrooms"].astype(float)).round(), np.nan)
df["is_multi_unit"] = df["bedrooms"].fillna(0) > 8

for col in ["listing_type", "property_type", "property_group", "property_category",
            "area", "region", "currency_original"]:
    df[col] = df[col].astype("category")

df.dtypes

listing_id                      str
listing_type               category
title                           str
property_type              category
location                        str
bedrooms                      Int64
bathrooms                     Int64
toilets                       Int64
parking                       Int64
url                             str
scraped_on           datetime64[us]
price_original              float64
currency_original          category
price_ngn                     int64
area                       category
neighbourhood                   str
region                     category
property_group             category
property_category          category
price_ngn_m                 float64
price_per_bedroom           float64
is_multi_unit               boolean
dtype: object

## Step 13: Flag outliers (not removed)
Very expensive listings in Ikoyi or Banana Island are **real**, so they are not deleted. Instead they are **flagged** with the IQR method so the EDA can include or exclude them.

Because property prices are hugely skewed, the IQR is calculated on the **log of price**, separately for Sale and Rent, so a ₦3M rent and a ₦3B sale are each judged against their own market.

In [16]:
def iqr_flag(s):
    log = np.log10(s)
    q1, q3 = log.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (log < q1 - 1.5 * iqr) | (log > q3 + 1.5 * iqr)

df["is_outlier"] = df.groupby("listing_type", observed=True)["price_ngn"].transform(iqr_flag)
print(df.groupby("listing_type", observed=True)["is_outlier"].sum())
df[df["is_outlier"]].sort_values("price_ngn", ascending=False)[
    ["title", "listing_type", "area", "price_ngn_m"]].head(8)

listing_type
Rent     8
Sale    23
Name: is_outlier, dtype: int64


,title,listing_type,area,price_ngn_m
887,Block of 36 Flats with 4 Bedrooms Each,Sale,Ikoyi,32000.0
335,"38,752SQM Parcel of Land",Sale,Ikeja,25000.0
987,Luxury 12 Units of 3 Bed Apartment and 1 Unit ...,Sale,Ikoyi,22644.0
432,Well Developed 5 Bedroom Detached House,Sale,Ikoyi,13320.0
821,Commercial Property,Sale,Ifako-Ijaiye,12500.0
78,A Well Built Industrial Warehouse + Office Block,Sale,Surulere,12000.0
168,"Land Measuring 4,100SQM [video Available on Re...",Sale,Ikeja,12000.0
928,About 70 Rooms Luxury Hotel,Sale,Victoria Island,10000.0


## Step 14: Final check and save

In [17]:
order = ["listing_id", "listing_type", "title", "property_type", "property_group", "property_category",
         "price_ngn", "price_ngn_m", "price_per_bedroom", "currency_original", "price_original",
         "bedrooms", "bathrooms", "toilets", "parking",
         "neighbourhood", "area", "region", "location",
         "is_multi_unit", "is_outlier", "url", "scraped_on"]
df = df[order].reset_index(drop=True)

print("Raw shape:  ", raw.shape)
print("Clean shape:", df.shape)
print("\nDuplicates left:", df.duplicated(subset="listing_id").sum())
print("\nMissing values (bedrooms/baths/toilets = land & commercial only):")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nSale vs Rent:")
print(df["listing_type"].value_counts())

Raw shape:   (2038, 18)
Clean shape: (1973, 23)

Duplicates left: 0

Missing values (bedrooms/baths/toilets = land & commercial only):
price_per_bedroom    183
bedrooms             183
bathrooms            174
toilets              161
dtype: int64

Sale vs Rent:
listing_type
Sale    1000
Rent     973
Name: count, dtype: int64


In [18]:
df.to_csv("lagos_properties_clean.csv", index=False)
print("Saved lagos_properties_clean.csv")
df.head()

Saved lagos_properties_clean.csv


,listing_id,listing_type,title,property_type,property_group,property_category,price_ngn,price_ngn_m,price_per_bedroom,currency_original,price_original,bedrooms,bathrooms,toilets,parking,neighbourhood,area,region,location,is_multi_unit,is_outlier,url,scraped_on
0,3388814,Sale,Newly Built 4 Bedroom Fully Detached Bungalow,Detached bungalow,Bungalow,Residential,100000000,100.0,25000000.0,NGN,100000000.0,4,4,5,3,Igando Orudu,Ibeju Lekki,Island,"Igando Orudu, Ibeju Lekki, Lagos",False,False,https://nigeriapropertycentre.com/for-sale/hou...,2026-10-10
1,3087107,Sale,Land,Residential land,Land,Land,50000000,50.0,NaN,NGN,50000000.0,<NA>,<NA>,<NA>,5,10Mins Drive To Command Off Ikola Road,Ipaja,Mainland,"Mainland Heritage, 10mins Drive to Command Off...",False,False,https://nigeriapropertycentre.com/for-sale/lan...,2026-10-10
2,3715040,Sale,Residential Land Plot Fenced,Residential land,Land,Land,75000000,75.0,NaN,NGN,75000000.0,<NA>,<NA>,<NA>,0,Alapere,Ketu,Mainland,"Off Ezobi Street, Alapere, Ketu, Lagos",False,False,https://nigeriapropertycentre.com/for-sale/lan...,2026-10-10
3,3663484,Sale,4 Bedroom Detached Duplex,Detached duplex,Duplex,Residential,280000000,280.0,70000000.0,NGN,280000000.0,4,4,5,0,Idado,Lekki,Island,"Idado, Lekki, Lagos",False,False,https://nigeriapropertycentre.com/for-sale/hou...,2026-10-10
4,3693891,Sale,Best Value 3 Bedroom Self Compound Terraced Du...,Terraced duplex,Duplex,Residential,115000000,115.0,38333333.0,NGN,115000000.0,3,3,4,2,Sangotedo,Ajah,Island,"Within a Gated, Very Secured Estate with No Fl...",False,False,https://nigeriapropertycentre.com/for-sale/hou...,2026-10-10


## Cleaning summary

**2,038 raw rows × 18 columns → 1,973 clean rows × 23 columns** (65 rows removed, 3.2%)

| Step | What was done | Rows affected |
|---|---|---|
| Duplicates | Removed near-duplicates (same title, price, location, sale/rent) | 24 removed |
| Currency | Text prices → numbers; fixed "$" on naira amounts; USD → naira at ₦1,332/$ | 2 fixed, 80 converted |
| Invalid prices | Removed per-sqm, monthly and misfiled prices | 39 removed |
| Location | Split into neighbourhood and area; standardised names; dropped non-locations | 2 removed |
| Bedrooms | Extracted from titles, then property-type median (residential only) | 47 + 21 filled |
| Bathrooms / toilets | Median for homes with the same bedroom count (residential only) | 47 / 42 filled |
| Parking | Missing → 0 ("not stated") | 1,038 filled |
| Land & commercial | Bedrooms, baths and toilets left empty on purpose (not applicable) | 183 rows |
| New features | region, property_group, property_category, price_ngn_m, price_per_bedroom, is_multi_unit | 6 columns added |
| Columns dropped | price_unit, size_sqm, added, page, raw price/currency | 7 columns dropped |
| Data types | Int64 counts, datetime, categories | all columns |
| Outliers | Flagged with log-IQR, not removed | 31 flagged |

**Known limitations**
- The sample is mostly Island listings (about 80%), because the site's first 50 pages favour promoted listings, which are concentrated in Lekki and Ikoyi.
- Rent is assumed to be annual. The scraper could not capture the price unit, so a few monthly prices above ₦200k may remain.
- USD listings use one exchange rate (₦1,332/$, 8 Oct 2026).